# Analisis Econometrico: Inflacion en Ecuador y el mecanismo de inflacion importada

Agente Econometrico/Estadistico. Reproduce de forma interactiva la logica de `scripts/econometric_model.py`
(estadistica descriptiva, correlacion de Pearson con p-valor, regresion lineal simple), sobre
`data/processed/inflacion_pib_desempleo.csv`.

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv('../data/processed/inflacion_pib_desempleo.csv')
inflacion = df[df['indicador'] == 'inflacion_precios_consumidor']
inflacion.head()

## Regresion lineal: inflacion de Ecuador ~ inflacion de Estados Unidos

Hipotesis a explorar: si el mecanismo de *inflacion importada* domina, se esperaria una pendiente
positiva y estadisticamente significativa.

In [ ]:
ecuador = inflacion[inflacion['pais'] == 'Ecuador'].set_index('anio')['valor']
eeuu = inflacion[inflacion['pais'] == 'Estados Unidos'].set_index('anio')['valor']
merged = pd.concat([ecuador.rename('ecuador'), eeuu.rename('eeuu')], axis=1).dropna()

slope, intercept, r_value, p_value, std_err = stats.linregress(merged['eeuu'], merged['ecuador'])
print(f"beta={slope:.3f}  intercepto={intercept:.3f}  R2={r_value**2:.3f}  p-valor={p_value:.4f}  n={len(merged)}")

**Interpretacion:** con datos anuales (n=11), el modelo no encuentra una relacion lineal fuerte ni
estadisticamente significativa (R2 muy bajo, p-valor >> 0.05). Esto no descarta el mecanismo de
inflacion importada -que opera principalmente sobre precios de bienes transables especificos, no
sobre el indice general- pero indica que, a esta frecuencia y tamano de muestra, no puede
confirmarse una transmision lineal directa y contemporanea. Ver discusion completa en
`docs/informe_final.pdf`, seccion 12.

## Correlacion inflacion-PIB e inflacion-desempleo, por pais

In [ ]:
pivot = df.pivot_table(index=['pais', 'anio'], columns='indicador', values='valor').reset_index()

rows = []
for pais, group in pivot.groupby('pais'):
    group = group.dropna()
    r_pib, p_pib = stats.pearsonr(group['inflacion_precios_consumidor'], group['crecimiento_pib'])
    r_des, p_des = stats.pearsonr(group['inflacion_precios_consumidor'], group['tasa_desempleo'])
    rows.append({
        'pais': pais,
        'corr_inflacion_pib': round(r_pib, 3),
        'p_valor_pib': round(p_pib, 3),
        'corr_inflacion_desempleo': round(r_des, 3),
        'p_valor_desempleo': round(p_des, 3),
    })

pd.DataFrame(rows)

**Interpretacion:** la unica correlacion estadisticamente significativa al 5% es la de Panama entre
inflacion y crecimiento del PIB (p < 0.05). El resto de coeficientes debe leerse como evidencia
exploratoria dado el tamano de muestra reducido (n=11 anos por pais). Ver interpretacion completa
en `docs/informe_final.pdf`, secciones 10.4 y 12.